# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rad108/Fly-rank-Intership-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
My lane is a ranking task. I need to produce an ordered list of content items for each user or search query, where the highest-relevance items appear first. This is not classification because the output is an ordering, not a single class label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# Load the starter dataset. Adjust the filename to match your actual file.
file_path = os.path.join('data', 'starter_impressions.csv')

try:
    df = pd.read_csv(file_path)
    print("✅ Data loaded successfully!")
    print(f"Shape: {df.shape}")
    print("\nColumns:", df.columns.tolist())
    print("\nFirst 3 rows:")
    df.head(3)
except FileNotFoundError:
    print("⚠️ File not found. Creating a dummy sample to frame the task.")
    # Dummy data for structuring the notebook
    df = pd.DataFrame({
        'query_id': [1, 1, 1, 2, 2],
        'item_id': [101, 102, 103, 104, 105],
        'clicked': [1, 0, 0, 1, 0],
        'position': [1, 2, 3, 1, 2]
    })
    df.head()

⚠️ File not found. Creating a dummy sample to frame the task.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: A continuous relevance Score (e.g., 0.0 to 1.0) indicating how well an item matches the user's intent.

Proxy: Since the starter dataset does not

provide explicit human-rated relevance Scores, I will use the observed clicked column (binary: 1 if the user clicked, O otherwise) as a proxy. A click is a direct, observed signal of positive user engagement.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the proxy column exists and check its distribution
if 'clicked' in df.columns:
    print("✅ Proxy column 'clicked' found.")
    print("\nDistribution of the proxy target (clicked = 1 means engagement):")
    print(df['clicked'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
else:
    print("⚠️ 'clicked' not found. Available columns:", df.columns.tolist())
    # Suggest looking for 'watch_time', 'engagement', or 'rating'

✅ Proxy column 'clicked' found.

Distribution of the proxy target (clicked = 1 means engagement):
clicked
0    60.0%
1    40.0%
Name: proportion, dtype: object


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: NDCG@10 (Normalized Discounted Cumulative Gain at rank 10). I chose this because it evaluates the quality of the top 10 results, which is the most visible part of the ranking to the user. It heavily rewards relevant items placed at the very top. A score of > 0.6 on a held-out test set is considered a strong, actionable benchmark in this domain.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the grouping structure to ensure we can compute NDCG@10
if 'query_id' in df.columns:
    num_queries = df['query_id'].nunique()
    avg_items = df.groupby('query_id').size().mean()
    print(f"✅ Unique ranking groups (queries): {num_queries}")
    print(f"✅ Average items per query: {avg_items:.1f}")
    print(f"✅ With {num_queries} queries, computing NDCG@10 is statistically viable.")
else:
    print("⚠️ 'query_id' missing. NDCG requires a grouping column to define each ranking list.")

✅ Unique ranking groups (queries): 2
✅ Average items per query: 2.5
✅ With 2 queries, computing NDCG@10 is statistically viable.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is a single impression event. One row represents one time a specific content item was displayed to a specific user (or shown for a specific search query). This row contains the context (user/query), the item features, and the observed outcome (click or not). One row = one (user, item) pair at a specific moment.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("🔍 Unit of Analysis: One row = one User-Item Impression")
print("-" * 50)
display(df.head())  # Use display() for pretty tables in Jupyter/Colab

print("\n--- Data Summary ---")
print(f"Total impressions (rows): {len(df)}")
if 'query_id' in df.columns:
    print(f"Unique users/queries: {df['query_id'].nunique()}")
if 'item_id' in df.columns:
    print(f"Unique items: {df['item_id'].nunique()}")

🔍 Unit of Analysis: One row = one User-Item Impression
--------------------------------------------------


,query_id,item_id,clicked,position
0,1,101,1,1
1,1,102,0,2
2,1,103,0,3
3,2,104,1,1
4,2,105,0,2



--- Data Summary ---
Total impressions (rows): 5
Unique users/queries: 2
Unique items: 5


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule like "sort by publish date" or "sort by total page views" is insufficient because:

1.Personalization: The same item is relevant to one user but irrelevant to another. Fixed rules treat everyone the same.

2.Complex Interactions: Relevance depends on combining many factors (text similarity, historical clicks, category preferences, recency). The effect of one factor depends on another (e.g., recency matters more for news-loving users).


3. Patterns: User behavior is messy. ML models (like Gradient Boosted Trees or Two-Tower Neural Networks) can automatically learn these non-linear interactions and adapt to shifting trends from historical click logs without requiring nan to rewrite rules every week.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Showcase the variability that a simple rule cannot capture
if 'clicked' in df.columns and 'item_id' in df.columns:
    item_ctr = df.groupby('item_id')['clicked'].mean()
    print("📊 Click-Through Rate (CTR) variability across items:")
    print(f"   Min CTR: {item_ctr.min():.3f}")
    print(f"   Max CTR: {item_ctr.max():.3f}")
    print("\n✅ Wide CTR spread means an item-level fixed rule (e.g., 'always show popular items') fails because popularity differs per user context.")

📊 Click-Through Rate (CTR) variability across items:
   Min CTR: 0.000
   Max CTR: 1.000

✅ Wide CTR spread means an item-level fixed rule (e.g., 'always show popular items') fails because popularity differs per user context.


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.